# 06 分组聚合与透视

数据分析里最高频的一句话是"**按××分组，算一下××**"：

- 每个城市的销售额是多少？
- 每个品类里最贵的 3 单是哪些？
- 每个销售代表占本城市业绩的比例？
- 各城市 × 各品类的交叉表？

这些都属于同一个模式：**split（拆）→ apply（算）→ combine（合）**。
本节把 `groupby` 的四种产出、`agg` 的写法、`transform` / `filter` 的用途一次讲清。

> 环境：`llm_sft`，从 `python/pandas` 目录启动。

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.max_columns", 30)

DATA = Path("data")
if not DATA.exists():
    DATA = Path("python/pandas/data")

orders = pd.read_csv(DATA / "sales_orders.csv", parse_dates=["order_date"])
orders.head(3)

,order_id,order_date,city,category,channel,unit_price,qty,amount,sales_rep
0,O001,2024-01-08,上海,食品,门店,95.9,4,383.6,王强
1,O002,2024-01-12,广州,家居,线上,453.4,4,1813.6,张伟
2,O003,2024-01-13,杭州,服饰,门店,120.7,3,362.1,王强


## 1. 心智模型：拆 → 算 → 合

```
原始表                按 city 拆成若干小组          每组算一个数        合回一张表
┌──────────┐          ┌─ 北京: 3 行 ─┐
│ 订单 ... │   groupby│─ 上海: 2 行 ─│  agg(sum)   →  city / total
└──────────┘          └─ 广州: 1 行 ─┘
```

`groupby` 只是"拆"，真正算什么是后面的 `agg` / `transform` / `filter` 决定的。

In [2]:
grouped = orders.groupby("city")
print("分成了几组:", grouped.ngroups)
print("每组的行数:", grouped.size().to_dict())
print("第一组的原始行（北京）:")
grouped.get_group("北京").head(3)

分成了几组: 6
每组的行数: {'上海': 7, '北京': 9, '广州': 12, '成都': 5, '杭州': 13, '深圳': 14}
第一组的原始行（北京）:


,order_id,order_date,city,category,channel,unit_price,qty,amount,sales_rep
3,O004,2024-01-16,北京,家居,线上,758.4,4,3033.6,赵敏
15,O016,2024-03-05,北京,家居,线上,199.3,5,996.5,李娜
19,O020,2024-03-19,北京,服饰,线下,70.6,2,141.2,李娜


## 2. 最常用：`groupby(...)[列].agg(函数)`

| 写法 | 返回 | 说明 |
| --- | --- | --- |
| `df.groupby("k")["v"].sum()` | Series（索引是 k） | 最简 |
| `df.groupby("k")["v"].sum().reset_index()` | DataFrame | 把索引变回列 |
| `df.groupby("k", as_index=False)["v"].sum()` | DataFrame | 同上，更直接 |
| `df.groupby(["k1", "k2"])["v"].sum()` | Series（两层索引） | 多列分组 |

In [3]:
print("Series 形态:")
print(orders.groupby("city")["amount"].sum())

Series 形态:
city
上海     9723.9
北京     9986.5
广州    10123.6
成都    14646.7
杭州    15879.8
深圳    11113.7
Name: amount, dtype: float64


In [4]:
print("DataFrame 形态（as_index=False）:")
orders.groupby("city", as_index=False)["amount"].sum().sort_values("amount", ascending=False)

DataFrame 形态（as_index=False）:


,city,amount
4,杭州,15879.8
3,成都,14646.7
5,深圳,11113.7
2,广州,10123.6
1,北京,9986.5
0,上海,9723.9


In [5]:
# 多列分组：结果自动变成两层索引
by_city_channel = orders.groupby(["city", "channel"])["amount"].sum()
print(by_city_channel.head(8))
print("\nunstack 一下更好读:")
by_city_channel.unstack(fill_value=0)

city  channel
上海  线上        910.5
      线下       7881.3
      门店        932.1
北京  线上       8425.2
      线下       1099.1
      门店        462.2
广州  线上       4452.6
      线下       4466.3
Name: amount, dtype: float64

unstack 一下更好读:


channel,线上,线下,门店
city,,,
上海,910.5,7881.3,932.1
北京,8425.2,1099.1,462.2
广州,4452.6,4466.3,1204.7
成都,5141.4,0.0,9505.3
杭州,12110.5,2735.8,1033.5
深圳,3021.3,2891.6,5200.8


## 3. `agg`：一次算多个指标

三种写法，从简单到灵活：

1. `agg("sum")` / `agg(["sum", "mean"])`：函数名；
2. `agg({"amount": "sum", "qty": "mean"})`：不同列用不同函数；
3. `agg(总金额=("amount", "sum"), 订单数=("order_id", "count"))`：**命名聚合**，列名直接写成中文，最推荐。

In [6]:
orders.groupby("city")["amount"].agg(["count", "sum", "mean", "max"]).round(1)

,count,sum,mean,max
city,,,,
上海,6,9723.9,1620.6,7109.0
北京,9,9986.5,1109.6,4368.5
广州,12,10123.6,843.6,2735.5
成都,5,14646.7,2929.3,9339.5
杭州,13,15879.8,1221.5,8886.4
深圳,12,11113.7,926.1,5068.0


In [7]:
# 不同列不同函数：注意列名会变成 MultiIndex
orders.groupby("city").agg({"amount": "sum", "qty": "mean"}).round(1)

,amount,qty
city,,
上海,9723.9,3.7
北京,9986.5,2.8
广州,10123.6,3.7
成都,14646.7,3.4
杭州,15879.8,3.2
深圳,11113.7,2.5


In [8]:
# 命名聚合：列名清晰，结果扁平，最适合直接出报表
summary = orders.groupby("city", as_index=False).agg(
    订单数=("order_id", "count"),
    销售额=("amount", "sum"),
    平均单价=("unit_price", "mean"),
    最大单笔=("amount", "max"),
)
summary["客单价"] = (summary["销售额"] / summary["订单数"]).round(1)
summary.sort_values("销售额", ascending=False).round(1)

,city,订单数,销售额,平均单价,最大单笔,客单价
4,杭州,13,15879.8,384.6,8886.4,1221.5
3,成都,5,14646.7,729.7,9339.5,2929.3
5,深圳,14,11113.7,391.7,5068.0,793.8
2,广州,12,10123.6,209.6,2735.5,843.6
1,北京,9,9986.5,300.7,4368.5,1109.6
0,上海,7,9723.9,409.5,7109.0,1389.1


In [9]:
# count 和 size 的区别：count 不数缺失，size 数所有行
print(orders.groupby("city")["sales_rep"].agg(["count", "size"]).head())

      count  size
city             
上海      7     7
北京      9     9
广州     12    12
成都      5     5
杭州     12    13


## 4. `transform`：算完还保持原来的行数

`agg` 把每组压成一行；`transform` 把每组的计算结果**广播回每一行**。用途：

- 组内占比：`每组销售额 / 该组总额`
- 组内离差：`本行 - 组均值`
- 组内标准化：`(本行 - 组均值) / 组标准差`
- 用组均值填充缺失

In [10]:
df = orders[["city", "amount"]].copy()
df["city_total"] = df.groupby("city")["amount"].transform("sum")
df["占比"] = (df["amount"] / df["city_total"]).round(3)
df["组内均值差"] = (df["amount"] - df.groupby("city")["amount"].transform("mean")).round(1)
df.head(6)

,city,amount,city_total,占比,组内均值差
0,上海,383.6,9723.9,0.039,-1237.0
1,广州,1813.6,10123.6,0.179,970.0
2,杭州,362.1,15879.8,0.023,-859.4
3,北京,3033.6,9986.5,0.304,1924.0
4,广州,403.0,10123.6,0.040,-440.6
5,杭州,2529.2,15879.8,0.159,1307.7


In [11]:
# 组内标准化（z-score）：比较"在本城市里算不算大单"
df["z"] = df.groupby("city")["amount"].transform(lambda s: (s - s.mean()) / s.std())
df.nlargest(3, "z")[["city", "amount", "z"]].round(2)

,city,amount,z
16,杭州,8886.4,3.20
40,深圳,5068.0,2.88
29,广州,2735.5,2.20


In [12]:
# 用组内中位数填充缺失
with_missing = orders[["city", "amount"]].copy()
with_missing.loc[with_missing.index[:4], "amount"] = np.nan
with_missing["filled"] = with_missing["amount"].fillna(
    with_missing.groupby("city")["amount"].transform("median")
)
with_missing.head(6)

,city,amount,filled
0,上海,NaN,720.90
1,广州,NaN,403.00
2,杭州,NaN,439.75
3,北京,NaN,273.15
4,广州,403.0,403.00
5,杭州,2529.2,2529.20


## 5. `filter`：按"组"筛选，返回原始行

与布尔筛选的区别：`filter` 的条件作用在**每个组**上，整组保留或整组丢弃。

In [13]:
big_cities = orders.groupby("city").filter(lambda g: len(g) >= 12)
print("保留的城市:", sorted(big_cities["city"].unique()))
print("行数变化:", len(orders), "→", len(big_cities))

保留的城市: ['广州', '杭州', '深圳']
行数变化: 60 → 39


## 6. 每组取 Top-N

套路只有一句：**先按组内指标排序，再 `groupby(...).head(n)`**。

In [14]:
top2 = (
    orders.sort_values(["city", "amount"], ascending=[True, False])
    .groupby("city")
    .head(2)
)
top2[["city", "order_id", "amount"]].head(10)

,city,order_id,amount
11,上海,O012,7109.0
54,上海,O055,910.5
46,北京,O047,4368.5
3,北京,O004,3033.6
29,广州,O030,2735.5
1,广州,O002,1813.6
58,成都,O059,9339.5
42,成都,O043,2677.8
16,杭州,O017,8886.4
5,杭州,O006,2529.2


In [15]:
# 每组只取"最大的一行"，用 idxmax 更快
idx = orders.groupby("city")["amount"].idxmax()
orders.loc[idx, ["city", "order_id", "amount"]]

,city,order_id,amount
11,上海,O012,7109.0
46,北京,O047,4368.5
29,广州,O030,2735.5
58,成都,O059,9339.5
16,杭州,O017,8886.4
40,深圳,O041,5068.0


## 7. 交叉表与透视表

- `crosstab`：**计数**专用，最像 Excel 的数据透视；
- `pivot_table`：可以指定 `values` 和 `aggfunc`，是透视的完整版。

In [16]:
pd.crosstab(orders["city"], orders["category"], margins=True, margins_name="合计")

category,家居,数码,服饰,食品,合计
city,,,,,
上海,0,2,3,2,7
北京,3,1,4,1,9
广州,5,0,3,4,12
成都,2,1,1,1,5
杭州,2,3,4,4,13
深圳,6,1,1,6,14
合计,18,8,16,18,60


In [17]:
pivot = orders.pivot_table(
    index="city",
    columns="category",
    values="amount",
    aggfunc="sum",
    fill_value=0,
    margins=True,
    margins_name="合计",
)
pivot.round(0)

category,家居,数码,服饰,食品,合计
city,,,,,
上海,0.0,7109.0,2180.0,435.0,9724.0
北京,4184.0,4368.0,1407.0,27.0,9986.0
广州,5904.0,0.0,3178.0,1041.0,10124.0
成都,4946.0,9340.0,166.0,196.0,14647.0
杭州,3287.0,9895.0,2042.0,655.0,15880.0
深圳,5715.0,5068.0,0.0,330.0,11114.0
合计,24037.0,35780.0,8973.0,2684.0,71474.0


In [18]:
# 同时看"销售额"和"订单数"：aggfunc 传列表
orders.pivot_table(
    index="channel",
    columns="category",
    values="amount",
    aggfunc=["sum", "count"],
    fill_value=0,
).round(0)

sum                          count               
category     家居     数码    服饰    食品  家居 数码 服饰 食品
channel                                                        
线上      16792.0  13743.0  2436.0  1091.0     9    3    2    8
线下       6289.0   7629.0  4873.0   283.0     5    2    8    5
门店        956.0  14408.0  1665.0  1310.0     3    2    5    5

## 8. 分箱统计：`cut` 与 `qcut`

连续值先分箱再 groupby，是看分布的标准动作：

- `cut`：按**你给的边界**分箱（等宽）；
- `qcut`：按**分位数**分箱（等频，每组样本数接近）。

In [19]:
orders["价格档"] = pd.cut(
    orders["unit_price"],
    bins=[0, 100, 500, 1500, 5000],
    labels=["<100", "100-500", "500-1500", ">1500"],
    right=False,
)
orders.groupby("价格档", observed=True).agg(
    订单数=("order_id", "count"), 销售额=("amount", "sum")
)

,订单数,销售额
价格档,,
<100,23,3501.0
100-500,20,15410.8
500-1500,15,34336.5
>1500,2,18225.9


In [20]:
orders["金额分位"] = pd.qcut(orders["amount"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])
orders.groupby("金额分位", observed=True)["amount"].agg(["count", "min", "max"]).round(0)

,count,min,max
金额分位,,,
Q1,15,17.0,154.0
Q2,14,166.0,488.0
Q3,14,514.0,1415.0
Q4,14,1525.0,9340.0


## 9. 小练习

用 `orders` 完成一张"城市经营报表"，每个城市一行，包含：

1. 订单数、销售额；
2. 卖得最好的品类（提示：先 groupby 两列求和，再取每组 idxmax）；
3. 该城市销售额占全国的比例（`transform` 或除以总和都行）。

In [21]:
city_cat = orders.groupby(["city", "category"], as_index=False)["amount"].sum()
best = city_cat.loc[city_cat.groupby("city")["amount"].idxmax(), ["city", "category", "amount"]]
best.columns = ["city", "最佳品类", "最佳品类销售额"]

report = orders.groupby("city", as_index=False).agg(
    订单数=("order_id", "count"), 销售额=("amount", "sum")
)
report = report.merge(best, on="city").assign(全国占比=lambda d: (d["销售额"] / d["销售额"].sum()).round(3))
report.sort_values("销售额", ascending=False).round(1)

,city,订单数,销售额,最佳品类,最佳品类销售额,全国占比
4,杭州,13,15879.8,数码,9894.6,0.2
3,成都,5,14646.7,数码,9339.5,0.2
5,深圳,14,11113.7,家居,5715.2,0.2
2,广州,12,10123.6,家居,5904.4,0.1
1,北京,9,9986.5,数码,4368.5,0.1
0,上海,7,9723.9,数码,7109.0,0.1


## 10. 小结与性能提示

**记住 5 个形态**

| 需求 | 写法 |
| --- | --- |
| 每组一个数 | `groupby(...).agg(...)` |
| 每组多个指标、中文列名 | 命名聚合 `agg(指标=("列", "函数"))` |
| 算完保持行数 | `groupby(...).transform(...)` |
| 按组筛选整组 | `groupby(...).filter(...)` |
| 每组 Top-N | `sort_values(...).groupby(...).head(n)` |

**性能与坑**

1. 分组键是 `category` 或分箱结果时，加 `observed=True`，否则会生成大量空组合（并触发警告）。
2. 能用 `agg` / `transform` 就不要用 `apply`：`apply` 是 Python 逐组循环，慢一个量级。
3. `count` 数非空值，`size` 数行数，缺失多时结果不同。
4. `transform` 要求返回与组等长的结果，返回标量会报错（那是 `agg` 的事）。
5. 多列分组的 Series 结果是 MultiIndex，用 `.unstack()` / `.reset_index()` 转成好读的表。